# Notebook 14 — Construindo uma API com FastAPI

Último notebook da trilha! Aqui eu **construo** uma API REST. Junto tudo o que
aprendi: funções, POO, dicionários, JSON, validação e testes.

Uso o **FastAPI**: moderno, rápido e que gera documentação automática.

## Conteúdo
- instalando o FastAPI;
- primeira rota;
- rotas com parâmetros;
- recebendo dados (POST) com validação (Pydantic);
- a API completa do projeto (CRUD de tarefas);
- como rodar e testar.

> As células deste notebook **mostram** o código da API. Para executá-la de
> verdade, use o terminal (instruções no fim) — uma API roda como um servidor,
> não dentro do notebook.

## 1. Instalação
No terminal, dentro do ambiente virtual:

```bash
pip install fastapi uvicorn
```

- **fastapi** — o framework;
- **uvicorn** — o servidor que coloca a API no ar.

## 2. A primeira rota
Uma API mínima. O decorador `@app.get("/")` diz: "quando alguém fizer um GET na
raiz, chame esta função".

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def raiz():
    return {"mensagem": "Olá, API!"}
```

Rodar: `uvicorn arquivo:app --reload` e abrir http://127.0.0.1:8000

## 3. Documentação automática 🎉
Uma das melhores coisas do FastAPI: ele gera sozinho uma documentação interativa.
Com a API rodando, acesse:

- **http://127.0.0.1:8000/docs** (Swagger UI)
- **http://127.0.0.1:8000/redoc** (ReDoc)

Dá para testar todas as rotas pelo navegador, sem escrever nada.

## 4. Parâmetros de rota e de query
```python
# parâmetro de rota: /tarefas/1
@app.get("/tarefas/{tarefa_id}")
def buscar(tarefa_id: int):
    return {"id": tarefa_id}

# parâmetro de query: /tarefas?concluida=true
@app.get("/tarefas")
def listar(concluida: bool | None = None):
    return {"filtro_concluida": concluida}
```
O FastAPI usa as **type hints** (notebook 04) para converter e validar os tipos
automaticamente. Se mandar `/tarefas/abc`, ele já responde com erro 422.

## 5. Recebendo dados com Pydantic
Para criar algo (POST), defino o formato esperado com um **modelo Pydantic**. O
FastAPI valida o corpo da requisição sozinho.

```python
from pydantic import BaseModel

class TarefaEntrada(BaseModel):
    titulo: str
    concluida: bool = False

@app.post("/tarefas")
def criar(tarefa: TarefaEntrada):
    return {"recebido": tarefa.titulo}
```
Se faltar o `titulo`, ou ele vier com tipo errado, o FastAPI retorna 422 com uma
mensagem clara — sem eu escrever nenhuma validação à mão.

## 6. A estrutura REST de um CRUD
A API completa do projeto (em `api/main.py`) é um CRUD de tarefas seguindo o
padrão REST:

| Método | Rota | O que faz |
|--------|------|-----------|
| GET | `/tarefas` | lista todas |
| GET | `/tarefas/{id}` | busca uma |
| POST | `/tarefas` | cria |
| PUT | `/tarefas/{id}` | atualiza |
| DELETE | `/tarefas/{id}` | remove |

Esse é o mesmo padrão que consumi no notebook 13 — agora visto por dentro.

## 7. Trecho central da API (api/main.py)
Abaixo, o coração da aplicação (o arquivo completo está na pasta `api/`):

```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="API de Tarefas")
tarefas: dict[int, dict] = {}
proximo_id = 1

class TarefaEntrada(BaseModel):
    titulo: str
    concluida: bool = False

@app.post("/tarefas", status_code=201)
def criar_tarefa(dados: TarefaEntrada):
    global proximo_id
    tarefa = {"id": proximo_id, **dados.model_dump()}
    tarefas[proximo_id] = tarefa
    proximo_id += 1
    return tarefa

@app.get("/tarefas/{tarefa_id}")
def buscar_tarefa(tarefa_id: int):
    if tarefa_id not in tarefas:
        raise HTTPException(status_code=404, detail="Tarefa não encontrada")
    return tarefas[tarefa_id]
```
Repare no `HTTPException`: é assim que devolvo um 404 — exatamente o status que
aprendi a tratar como cliente no notebook 13.

## 8. Como rodar e testar a API do projeto
No terminal, a partir da raiz do repositório:

```bash
# instalar tudo
pip install -r requirements.txt

# subir a API (docs em http://127.0.0.1:8000/docs)
uvicorn api.main:app --reload

# rodar os testes automatizados da API
python -m unittest discover -s api -p "test_*.py" -v
```

A API é testada com o **TestClient** do FastAPI, que simula requisições sem
precisar subir o servidor — unindo este notebook com o notebook 12 (testes).

## Conclusão da trilha 🏁
Cheguei ao fim: saí dos fundamentos (notebook 01) e cheguei a **construir uma API
REST testada**.

### O que aprendi no caminho
fundamentos → controle de fluxo → estruturas de dados → funções → strings →
erros → POO → módulos → arquivos → código pythonic → biblioteca padrão →
testes → consumir APIs → **construir APIs**.

### Próximos passos
- conectar a API a um **banco de dados** (SQLite/PostgreSQL);
- **autenticação** (tokens JWT);
- **deploy** (colocar a API no ar, ex.: Render/Railway/Fly.io);
- estudar **assíncrono** (`async`/`await`) a fundo.